In [11]:
# (Same as Experiment 2 — re-run if starting a fresh notebook)
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

data = pd.read_csv("/content/mushrooms.csv")

encoders = {}
encoded_data = data.copy()
for col in encoded_data.columns:
    enc = LabelEncoder()
    encoded_data[col] = enc.fit_transform(encoded_data[col])
    encoders[col] = enc

X_full = encoded_data.drop(columns=["class"])
y_full = encoded_data["class"]

X_train, X_test, y_train, y_test = train_test_split(X_full, y_full, test_size=0.2, random_state=42)

In [3]:
def compute_entropy(labels):
    values, counts = np.unique(labels, return_counts=True)
    probs = counts / counts.sum()
    return -np.sum(probs * np.log2(probs))


def compute_gini(labels):
    """Gini(S) = 1 - sum(p_i^2)"""
    values, counts = np.unique(labels, return_counts=True)
    probs = counts / counts.sum()
    return 1 - np.sum(probs ** 2)


def compute_misclassification_error(labels):
    """ME(S) = 1 - max(p_i)"""
    values, counts = np.unique(labels, return_counts=True)
    probs = counts / counts.sum()
    return 1 - np.max(probs)

In [4]:
def compute_info_gain(feature_col, labels):
    total_entropy = compute_entropy(labels)
    values, counts = np.unique(feature_col, return_counts=True)
    weighted_entropy = 0
    for v, c in zip(values, counts):
        subset_labels = labels[feature_col == v]
        weighted_entropy += (c / len(labels)) * compute_entropy(subset_labels)
    return total_entropy - weighted_entropy


def compute_split_info(feature_col):
    """SplitInfo(A) = -sum((|Sv|/|S|) * log2(|Sv|/|S|)) over values v of feature A"""
    values, counts = np.unique(feature_col, return_counts=True)
    probs = counts / counts.sum()
    # avoid log2(0)
    return -np.sum(probs * np.log2(probs))


def compute_gain_ratio(feature_col, labels):
    """GainRatio(A) = InfoGain(A) / SplitInfo(A)"""
    ig = compute_info_gain(feature_col, labels)
    si = compute_split_info(feature_col)
    if si == 0:
        return 0
    return ig / si

In [5]:
def compute_gini_gain(feature_col, labels):
    """GiniGain(A) = Gini(S) - sum((|Sv|/|S|) * Gini(Sv))"""
    total_gini = compute_gini(labels)
    values, counts = np.unique(feature_col, return_counts=True)
    weighted_gini = 0
    for v, c in zip(values, counts):
        subset_labels = labels[feature_col == v]
        weighted_gini += (c / len(labels)) * compute_gini(subset_labels)
    return total_gini - weighted_gini

In [6]:
def compute_me_gain(feature_col, labels):
    """MEGain(A) = ME(S) - sum((|Sv|/|S|) * ME(Sv))"""
    total_me = compute_misclassification_error(labels)
    values, counts = np.unique(feature_col, return_counts=True)
    weighted_me = 0
    for v, c in zip(values, counts):
        subset_labels = labels[feature_col == v]
        weighted_me += (c / len(labels)) * compute_misclassification_error(subset_labels)
    return total_me - weighted_me

In [7]:
class TreeNode:
    def __init__(self, feature=None, branches=None, prediction=None):
        self.feature = feature
        self.branches = branches if branches is not None else {}
        self.prediction = prediction


CRITERIA = {
    "gain_ratio": compute_gain_ratio,
    "gini": compute_gini_gain,
    "misclassification": compute_me_gain,
}


def build_tree(X, y, criterion="gain_ratio"):
    score_fn = CRITERIA[criterion]

    if len(y.unique()) == 1:
        return TreeNode(prediction=y.iloc[0])
    if X.shape[1] == 0:
        return TreeNode(prediction=y.mode()[0])

    best_feature, best_score = None, -np.inf
    for col in X.columns:
        s = score_fn(X[col], y)
        if s > best_score:
            best_score, best_feature = s, col

    node = TreeNode(feature=best_feature)
    for val in np.unique(X[best_feature]):
        mask = X[best_feature] == val
        X_subset = X[mask].drop(columns=[best_feature])
        y_subset = y[mask]
        if len(y_subset) == 0:
            node.branches[val] = TreeNode(prediction=y.mode()[0])
        else:
            node.branches[val] = build_tree(X_subset, y_subset, criterion=criterion)

    return node


def predict_one(node, row):
    if node.prediction is not None:
        return node.prediction
    value = row[node.feature]
    if value not in node.branches:
        return 0
    return predict_one(node.branches[value], row)

In [8]:
results = {}

for criterion in ["gain_ratio", "gini", "misclassification"]:
    root = build_tree(X_train, y_train, criterion=criterion)
    preds = [predict_one(root, X_test.iloc[i]) for i in range(len(X_test))]
    acc = accuracy_score(y_test, preds)
    results[criterion] = {"root_node": root, "accuracy": acc, "predictions": preds}
    print(f"[{criterion}] Root feature: '{root.feature}' | Accuracy: {acc*100:.2f}%")

[gain_ratio] Root feature: 'odor' | Accuracy: 100.00%
[gini] Root feature: 'odor' | Accuracy: 100.00%
[misclassification] Root feature: 'odor' | Accuracy: 100.00%


In [9]:
comparison_rows = []
for col in X_train.columns:
    comparison_rows.append({
        "Feature": col,
        "GainRatio": compute_gain_ratio(X_train[col], y_train),
        "GiniGain": compute_gini_gain(X_train[col], y_train),
        "MEGain": compute_me_gain(X_train[col], y_train),
    })

comparison_df = pd.DataFrame(comparison_rows)

print("Best feature (root) by Gain Ratio:", comparison_df.sort_values("GainRatio", ascending=False).iloc[0]["Feature"])
print("Best feature (root) by Gini Gain :", comparison_df.sort_values("GiniGain", ascending=False).iloc[0]["Feature"])
print("Best feature (root) by ME Gain   :", comparison_df.sort_values("MEGain", ascending=False).iloc[0]["Feature"])

comparison_df.sort_values("GainRatio", ascending=False).head(10)

Best feature (root) by Gain Ratio: odor
Best feature (root) by Gini Gain : odor
Best feature (root) by ME Gain   : odor


,Feature,GainRatio,GiniGain,MEGain
4,odor,0.391861,0.471113,0.467610
7,gill-size,0.268229,0.150461,0.242345
11,stalk-surface-above-ring,0.237026,0.176013,0.259886
19,spore-print-color,0.218206,0.282157,0.349592
18,ring-type,0.205856,0.180807,0.256501
3,bruises,0.199697,0.127518,0.228035
12,stalk-surface-below-ring,0.197352,0.167057,0.250346
6,gill-spacing,0.154976,0.059536,0.097092
8,gill-color,0.138094,0.231471,0.285890
13,stalk-color-above-ring,0.131192,0.138012,0.198646


In [10]:
for criterion, res in results.items():
    print(f"\n--- {criterion} ---")
    print(f"Accuracy: {res['accuracy']*100:.2f}%")
    print(confusion_matrix(y_test, res['predictions']))


--- gain_ratio ---
Accuracy: 100.00%
[[843   0]
 [  0 782]]

--- gini ---
Accuracy: 100.00%
[[843   0]
 [  0 782]]

--- misclassification ---
Accuracy: 100.00%
[[843   0]
 [  0 782]]
